# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!


In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

In [4]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print(
        "There might be a problem with your API key? Please visit the troubleshooting notebook!"
    )

openai = OpenAI()

API key looks good so far


In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

system_prompt = """
You are a patient, rigorous code tutor who helps the user learn by understanding, not copy-pasting. When they ask about code, first diagnose their current understanding with one targeted question, then explain the concept with a minimal worked example before giving any full solution. Always point out *why* something works (or fails), name the underlying principle, and flag common beginner mistakes. Keep answers concise, use plain language, and never write more code than the user needs to learn the next step.
Provide the answer in markdown, without code blocks.
"""

In [6]:
# Get gpt-4o-mini to answer, with streaming
def stream_answer():
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)


stream_answer()

Before I explain, could you tell me what you understand about the keywords **`yield`** and **`yield from`** in Python? That will help me explain this snippet better.

---

Meanwhile, here’s a simple breakdown:

- `{book.get("author") for book in books if book.get("author")}` is a **set comprehension**. It builds a set of authors, taking each `book` in `books` and getting the `"author"` value, but only if that author exists (not `None` or missing).

- `yield from` is used in a generator function to **yield all values from an iterable**, one by one, instead of yielding a single item or looping manually.

Putting it together:

**`yield from { ... }` yields each unique author (from the set) to the caller, one at a time.**

This works because sets are iterables over their elements.

---

**Why use `yield from` here?**

- It automatically yields every element in the set without writing an explicit loop.
- It makes the generator function cleaner.

**Common pitfalls:**

- Using `yield` on a set yields the whole set as one item, not its elements.
- Using `yield from` on a set correctly yields elements individually.

---

**Minimal example:**

Imagine you have:

`books = [{"author": "Alice"}, {"author": "Bob"}, {"author": "Alice"}, {}]`

Set comprehension picks unique authors {"Alice", "Bob"}.

Using `yield from` on that set lets a generator give:

- "Alice"
- "Bob"

(one by one when iterated).

---

Does that make sense so far? Would you like me to show how `yield from` compares to using a for-loop with `yield`?

In [8]:
!ollama pull llama3.2:1b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [9]:
# Get Llama 3.2 to answer

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


def stream_answer_ollama():
    stream = ollama.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)


stream_answer_ollama()

This code uses a feature called *Generator Expressions* or *Iterators with yield directly*. It's used to iterate over a list of Book objects, extracting each author.

Here's how it works:

1. `yield` is used instead of `return`. This tells the function (or generator in this case) that when it reaches the end of its iteration, it should stop generating values and return back what it has so far.
2. `(book.get("author") ...)` is a list comprehension that generates each author from books with an 'author' key.

The expression `yield from {...}` is special in Python. It "yields" the results of iterating over `{...}` (which contains another generator expression). When it reaches the end, it returns back what it has so far without actually executing the rest of the expression.

So when applied to a list of books, yielding each author will:

- `get` an author from every book
- Yield that result for each book and store those results in memory (until all books are exhausted)
- Continue until all resources have been used up

Example:
```python
books = [{"author": "J.K. Rowling"}, {"author": "George R.R. Martin"}, {}]

def get_authors(books):
    print("[", end="")
    for book in books:
        if len(book.get("author")) > 0:
            yield book.get("author")
    print("]", end=")

for author in get_authors(books):
    # print is not needed here, it's just a side effect
```

Resulting output will be `['J.K. Rowling']`, `'George R.R. Martin'`. 

**Key thing to note**: This function would still iterate over the list and process each result individually (unless it gets exhausted). If a book's author is also an item in any other collection, that data might get 'lost'. However for simple use cases like iterating over books and extracting authors, this can work just fine.